<a href="https://colab.research.google.com/github/bermanlabemory/unsupervised-behavior-tutorials/blob/main/00_colab_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Colab check &mdash; run this *before* the workshop

**Time: about 2 minutes.** This short notebook doesn't teach anything; it just confirms that your
Google Colab runtime can do everything we'll ask of it today. Run every
cell top to bottom (Shift+Enter, or `Runtime → Run all`). If the last cell prints a happy message,
you're set. If something breaks, let me know!

# 1.&nbsp; What machine did Google give us?

In [ ]:
import sys, platform, subprocess
print("Python:", sys.version.split()[0], "on", platform.system())

# A GPU is optional for this workshop, but nice to have -- it speeds up UMAP and the autoencoders.
# To ask for one: Runtime -> Change runtime type -> Hardware accelerator -> GPU, then rerun this cell.
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                     capture_output=True, text=True)
print("GPU:", gpu.stdout.strip() if gpu.returncode == 0 else "none (that's perfectly fine)")

# 2.&nbsp; Grab the analysis package

We'll lean on **motionmapperpy** all afternoon &mdash; a GPU-friendly Python implementation of
MotionMapper. The cell below clones it from GitHub (which also brings the small example datasets),
installs the handful of packages Colab doesn't already ship, and imports it. The same setup cell
opens every notebook in this series, so this is your first look at it.

In [ ]:
import os, sys, types
if not os.path.exists("motionmapperpy"):
    !git clone -q https://github.com/bermanlabemory/motionmapperpy

# motionmapperpy imports moviepy the moment it loads, and Colab's moviepy/ffmpeg stack tries to
# fetch an ancient ffmpeg from a (long-dead) URL. None of these notebooks push video through moviepy,
# so we hand Python a tiny stand-in for it -- harmless here, and it sidesteps the whole mess.
def _stub(name, **attrs):
    m = sys.modules.get(name) or types.ModuleType(name)
    for k, v in attrs.items():
        setattr(m, k, v)
    sys.modules[name] = m
    return m
_stub("moviepy"); _stub("moviepy.editor", VideoClip=object, VideoFileClip=object)
_stub("moviepy.video"); _stub("moviepy.video.io")
_stub("moviepy.video.io.bindings", mplfig_to_npimage=lambda *a, **k: None)

# We import motionmapperpy straight from the cloned folder. (Running its setup.py on Colab leaves an
# importable-but-empty package -- the classic trap; this route avoids it, and needs no kernel restart.)
sys.path.insert(0, os.path.abspath("motionmapperpy"))
for _m in [k for k in list(sys.modules) if k.startswith("motionmapperpy")]:
    del sys.modules[_m]

!pip install -q hdf5storage easydict umap-learn 2>/dev/null

import numpy as np
import matplotlib.pyplot as plt
import motionmapperpy as mmpy

print("engine ready")

> **No restart needed.** This cell imports motionmapperpy straight from the cloned folder. If
> `import motionmapperpy` ever fails, just re-run the cell &mdash; do **not** use *Restart session*,
> which would undo the `sys.path` line it adds. Your files are kept either way.

# 3.&nbsp; Test on fake data

One real step of the pipeline on fake data. A **wavelet transform** is core to the methods's approach &mdash;
it turns a posture time series into a picture of *how fast things are wiggling, at each moment*
(you'll see why that matters in notebook 01). If the cell below draws a spectrogram and prints a
check mark, every piece we need is working.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import motionmapperpy as mmpy

params = mmpy.setRunParameters()
params.samplingFreq, params.minF, params.maxF, params.numPeriods = 100, 1, 50, 25

# A tiny fake "postural" time series: 3 modes drifting over 2000 frames. If the wavelet transform
# returns an array we can plot, the whole engine runs on this machine.
x = np.cumsum(np.random.randn(2000, 3), axis=0)
wavelets, freqs = mmpy.findWavelets(x, 3, params.omega0, params.numPeriods, params.samplingFreq,
                                    params.maxF, params.minF, params.numProcessors, -1)

fig, ax = plt.subplots(figsize=(9, 3))
ax.imshow(wavelets[:500].T, aspect="auto", cmap="PuRd", origin="lower")
ax.set_title("a wavelet spectrogram of fake data — if you can see this, you're ready")
ax.set_xlabel("frame"); ax.set_ylabel("wavelet channel"); plt.show()

print("\n✅ All set!")